In [21]:
# -*- coding: utf-8 -*-
"""
LinkedIn Post Helper - Cell 1: Setup, Configuration, and Functions

Installs libraries, imports modules, gets LinkedIn App credentials,
defines API info & functions, and initializes variables. Defaults posts
to 'CONNECTIONS' visibility.

**ACTION REQUIRED:** Have your LinkedIn App's Client ID, Client Secret,
and the exact Authorized Redirect URI ready before running this cell.
(Recommended Redirect URI: https://www.linkedin.com/developers/tools/oauth/redirect)
Ensure the required LinkedIn App Products ('Sign In...', 'Share on LinkedIn')
are approved.
"""

# --- 1. Install Libraries ---
print("Initializing...")
!pip install requests -q
print("✅ Libraries checked.")

# --- 2. Import Modules ---
import requests
import json
from urllib.parse import urlencode, urlparse, parse_qs
import uuid
import getpass
from IPython import get_ipython
print("✅ Modules imported.")

# --- 3. Get App Credentials from User ---
print("-" * 30)
print("Please provide your LinkedIn Application details:")
CLIENT_ID = input("Enter your LinkedIn App Client ID: ")
CLIENT_SECRET = getpass.getpass("Enter your LinkedIn App Client Secret (input hidden): ")
REDIRECT_URI = input("Enter your EXACT LinkedIn App Redirect URI: ")
print("✅ Credentials received.")

# --- 4. Define API Info ---
AUTHORIZATION_URL = 'https://www.linkedin.com/oauth/v2/authorization'
ACCESS_TOKEN_URL = 'https://www.linkedin.com/oauth/v2/accessToken'
USER_INFO_URL = 'https://api.linkedin.com/v2/userinfo' # Using userinfo endpoint
UGC_POST_URL = 'https://api.linkedin.com/v2/ugcPosts'
SCOPES = 'openid profile email w_member_social'
LINKEDIN_API_VERSION = '202310' # Use recent YYYYMM format
print(f"✅ API Info Defined. Requesting Scopes: {SCOPES}. Using API Version: {LINKEDIN_API_VERSION}")

# --- 5. Initialize Session State Variables ---
access_token = None
user_urn = None
state_value_to_send = None # Variable name used for %store
auth_code_from_redirect = None # Will be set by Cell 3
state_from_redirect = None   # Will be set by Cell 3
print("✅ Session variables initialized.")

# --- 6. Define Helper Functions ---

def handle_token_expiry(error_context="Token Error", force_clear=False):
    """ Clears session state and guides user to restart auth."""
    global access_token, user_urn
    if access_token or force_clear:
        print("\n" + "=" * 60)
        print(f"⚠️ {error_context.upper()} DETECTED / SESSION RESET!")
        print("   Clearing token and user information.")
        access_token = None; user_urn = None
        print("   Session state cleared.")
        print("\n➡️ ACTION REQUIRED: Restart authentication from Cell 2.")
        print("=" * 60 + "\n")

def fetch_linkedin_user_urn():
    """ Fetches identifier from /v2/userinfo and constructs the person URN."""
    global user_urn
    if not access_token: print("   ❌ Cannot fetch URN: Access token missing."); return False
    print("   Fetching User Info via /v2/userinfo...")
    headers = {'Authorization': f'Bearer {access_token}', 'Connection': 'Keep-Alive', 'Content-Type': 'application/json', 'X-Restli-Protocol-Version': '2.0.0', 'LinkedIn-Version': LINKEDIN_API_VERSION}
    try:
        response = requests.get(USER_INFO_URL, headers=headers)
        req_id = response.headers.get('x-li-uuid', 'N/A')
        if response.status_code == 401: print(f"   ❌ 401 Unauthorized fetching URN (Req ID: {req_id}). Token expired?"); handle_token_expiry("Token Error Fetching URN", True); return False
        if response.status_code == 403: print(f"   ❌ 403 Forbidden fetching URN (Req ID: {req_id}). Response: {response.text}"); print("      Hint: Check 'profile'/'openid' scope/product."); handle_token_expiry("Permission Error Fetching URN", True); return False
        response.raise_for_status()
        user_data = response.json()
        subject_id = user_data.get('sub')
        if not subject_id: print(f"   ❌ Error: Could not extract 'sub' from UserInfo response: {user_data}"); handle_token_expiry("Bad UserInfo Response", True); return False
        constructed_urn = f"urn:li:person:{subject_id}"
        user_urn = constructed_urn
        print(f"   ✅ User URN constructed: {user_urn}")
        return True
    except requests.exceptions.RequestException as e: print(f"   ❌ Network/Request Error fetching URN: {e}"); handle_token_expiry("API Request Failed", True); return False

def exchange_code_for_token(auth_code_to_exchange, state_to_verify):
    """ Exchanges auth code for token, verifies state, fetches URN."""
    global access_token, user_urn
    print("   Verifying state...")
    original_state = None; ipython = get_ipython()
    if not ipython: print("   🔴 Error: Cannot get IPython instance to verify state."); return False
    try:
        ipython.run_line_magic('store', '-r state_value_to_send')
        if 'state_value_to_send' in locals() or 'state_value_to_send' in globals(): original_state = state_value_to_send
        else: raise KeyError("state_value_to_send not found after %store -r")
    except Exception as e: print(f"   🔴 Error retrieving stored state: {e}. Restart Cell 2."); return False

    if state_to_verify != original_state:
        print("   🔴 Error: State mismatch! Potential security risk. Aborting.");
        try: ipython.run_line_magic('store', '-d state_value_to_send');
        except Exception: pass
        return False
    else:
        print("   ✅ State verified.")
        try: ipython.run_line_magic('store', '-d state_value_to_send');
        except Exception: pass

    token_payload = {'grant_type': 'authorization_code', 'code': auth_code_to_exchange, 'redirect_uri': REDIRECT_URI, 'client_id': CLIENT_ID, 'client_secret': CLIENT_SECRET}
    token_headers = {'Content-Type': 'application/x-www-form-urlencoded'}
    print("   Requesting access token...")
    try:
        response = requests.post(ACCESS_TOKEN_URL, data=token_payload, headers=token_headers)
        response.raise_for_status()
        token_data = response.json()
        new_access_token = token_data.get('access_token')
        if not new_access_token: print(f"   🔴 Error: Access token not found in response. Body: {response.text}"); return False
        access_token = new_access_token
        print(f"   ✅ Access Token retrieved!")
        if not fetch_linkedin_user_urn(): return False # Fetch URN immediately
        return True # Success
    except requests.exceptions.RequestException as e:
        print(f"   🔴 Error during token request: {e}")
        if hasattr(e, 'response') and e.response is not None:
             print(f"      Status: {e.response.status_code}, Body: {e.response.text}")
             if e.response.status_code == 401: print("      Hint: Check Client ID/Secret.")
             if e.response.status_code == 400: print("      Hint: Check Redirect URI match / code validity.")
        return False

def post_linkedin_update(text_content):
    """ Posts a text update using the global token/URN. Defaults to CONNECTIONS visibility."""
    if not access_token or not user_urn: print("🔴 Cannot post: Auth state incomplete."); handle_token_expiry("Post Attempt Failed - State Missing", True); return False
    if not isinstance(user_urn, str) or not user_urn.startswith("urn:li:person:"): print(f"🔴 Cannot post: Invalid URN format: '{user_urn}'."); handle_token_expiry("Post Attempt Failed - Invalid URN", True); return False

    print(f"   Posting update (Visibility: CONNECTIONS)...")
    headers = {'Authorization': f'Bearer {access_token}', 'Content-Type': 'application/json', 'X-Restli-Protocol-Version': '2.0.0', 'LinkedIn-Version': LINKEDIN_API_VERSION}
    visibility_setting = {"com.linkedin.ugc.MemberNetworkVisibility": "CONNECTIONS"}
    post_body = {"author": user_urn, "lifecycleState": "PUBLISHED", "specificContent": {"com.linkedin.ugc.ShareContent": {"shareCommentary": {"text": text_content}, "shareMediaCategory": "NONE" }}, "visibility": visibility_setting}
    try:
        response = requests.post(UGC_POST_URL, headers=headers, json=post_body)
        req_id = response.headers.get('x-li-uuid', 'N/A')
        if response.status_code == 401: print(f"   🔴 401 Unauthorized post (Req ID: {req_id}). Token expired?"); handle_token_expiry("Token Error During Post"); return False
        if response.status_code == 403: print(f"   🔴 403 Forbidden post (Req ID: {req_id}). Response: {response.text}"); handle_token_expiry("Permission Error During Post"); return False
        response.raise_for_status()
        if response.status_code == 201:
            post_id = response.headers.get('x-restli-id', 'N/A')
            if post_id and post_id.startswith("urn:li:share:"): post_url = f"https://www.linkedin.com/feed/update/{post_id}/"; print(f"✅ Successfully posted!\n   View Post: {post_url}")
            else: print(f"✅ Successfully posted! (Post ID: {post_id})")
            return True
        else: print(f"⚠️ Unexpected post status {response.status_code}."); return False
    except requests.exceptions.RequestException as e:
        print(f"🔴 Network/Request Error posting: {e}")
        if hasattr(e, 'response') and e.response is not None:
             print(f"   Status: {e.response.status_code}, Body: {e.response.text}")
             if e.response.status_code == 422: print("   Hint: Check post content length or for policy violations.")
        return False

print("-" * 30)
print("✅ Setup and functions defined. Proceed to Cell 2.")

Initializing...
✅ Libraries checked.
✅ Modules imported.
------------------------------
Please provide your LinkedIn Application details:
Enter your LinkedIn App Client ID: 7825j3bkacjbx0
Enter your LinkedIn App Client Secret (input hidden): ··········
Enter your EXACT LinkedIn App Redirect URI: https://www.linkedin.com/developers/tools/oauth/redirect
✅ Credentials received.
✅ API Info Defined. Requesting Scopes: openid profile email w_member_social. Using API Version: 202310
✅ Session variables initialized.
------------------------------
✅ Setup and functions defined. Proceed to Cell 2.


In [22]:
# -*- coding: utf-8 -*-
"""
LinkedIn Post Helper - Cell 2: Generate Authorization URL

Generates the unique URL for the user to visit on LinkedIn to grant permissions.
Stores a 'state' value for security verification in Cell 4.
"""
# Make state variable global for %store
global state_value_to_send

# Pre-check: Ensure Cell 1 ran
if 'CLIENT_ID' not in globals() or not CLIENT_ID:
    print("🔴 Error: CLIENT_ID not defined. Please run Cell 1 first.")
else:
    print("STEP 1: Generate Authorization Link")
    print("-" * 60)
    print("Generating LinkedIn Authorization URL...")
    state_value_to_send = str(uuid.uuid4())
    # Store state using %store for secure verification after redirect
    %store state_value_to_send
    # print(f"   DEBUG: Stored state: {state_value_to_send}") # Uncomment if needed

    auth_params = {
        'response_type': 'code', 'client_id': CLIENT_ID,
        'redirect_uri': REDIRECT_URI, 'state': state_value_to_send, 'scope': SCOPES
    }
    auth_url = f"{AUTHORIZATION_URL}?{urlencode(auth_params)}"
    print("✅ Authorization URL generated.")
    print("\nACTION REQUIRED:")
    print(" 1. Click the link below.")
    print(" 2. Log in to LinkedIn & Click 'Allow' (if prompted).")
    print(" 3. After redirect, copy the **ENTIRE URL** from your browser's address bar.")
    print("\nAuthorization URL (Click Here):\n")
    print(auth_url)
    print("\n" + "-" * 60)
    print("➡️ After copying the URL, run Cell 3 and paste it when prompted.")
    print("-" * 60)

STEP 1: Generate Authorization Link
------------------------------------------------------------
Generating LinkedIn Authorization URL...
Stored 'state_value_to_send' (str)
✅ Authorization URL generated.

ACTION REQUIRED:
 1. Click the link below.
 2. Log in to LinkedIn & Click 'Allow' (if prompted).
 3. After redirect, copy the **ENTIRE URL** from your browser's address bar.

Authorization URL (Click Here):

https://www.linkedin.com/oauth/v2/authorization?response_type=code&client_id=7825j3bkacjbx0&redirect_uri=https%3A%2F%2Fwww.linkedin.com%2Fdevelopers%2Ftools%2Foauth%2Fredirect&state=5f3638c6-43f0-41b3-b75c-4be5c4883601&scope=openid+profile+email+w_member_social

------------------------------------------------------------
➡️ After copying the URL, run Cell 3 and paste it when prompted.
------------------------------------------------------------


In [23]:
# -*- coding: utf-8 -*-
"""
LinkedIn Post Helper - Cell 3: Process Redirected URL

User pastes the full URL from the browser after authorization. This cell
extracts the temporary 'code' and 'state' parameters needed for Cell 4.
"""
# Make variables global so Cell 4 can access them
global auth_code_from_redirect, state_from_redirect

# Pre-check: Ensure Cell 2 ran and generated state
if 'state_value_to_send' not in globals() or not state_value_to_send:
     print("🔴 Error: State value not generated. Please run Cell 2 first.")
else:
    print("STEP 1 (Continued): Provide Redirected URL")
    print("-" * 60)
    print("Waiting for the result from your browser action...")
    redirected_url = input("Paste the FULL redirected URL here: ")

    # Reset potentially stale values
    auth_code_from_redirect = None
    state_from_redirect = None

    # Safely parse the URL
    try:
        parsed_url = urlparse(redirected_url)
        query_params = parse_qs(parsed_url.query)
        # Store extracted values in global variables for Cell 4
        auth_code_from_redirect = query_params.get('code', [None])[0]
        state_from_redirect = query_params.get('state', [None])[0]

        if auth_code_from_redirect and state_from_redirect:
            print("\n✅ Authorization code and state extracted successfully.")
            print("\n➡️ Proceed to Cell 4 to complete authentication.")
        else:
            # Clear globals if extraction failed
            auth_code_from_redirect = None
            state_from_redirect = None
            if not auth_code_from_redirect: print("🔴 Error: 'code' not found in pasted URL.")
            if not state_from_redirect: print("🔴 Error: 'state' not found in pasted URL.")
            print("   Cannot proceed. Re-run Cell 2 & 3 ensure the full URL is pasted.")

    except Exception as e:
        print(f"🔴 Error parsing the URL: {e}")
        auth_code_from_redirect = None # Clear on error
        state_from_redirect = None   # Clear on error
        print("   Cannot proceed. Re-run Cell 2 & 3.")

    print("-" * 60)

STEP 1 (Continued): Provide Redirected URL
------------------------------------------------------------
Waiting for the result from your browser action...
Paste the FULL redirected URL here: https://www.linkedin.com/developers/tools/oauth/redirect?code=AQTdIebGvpyTCZFJWAwZ6zlQCIyOWTMhV1iX5tJ76ipDx-mkiZFqwXO75Huowum994sym3XuG8Mvml0XD59Om5abVqJII-eJOj0i8brF41nYn076yLM3i9QTkK3GV3yd8EA-C6u0CriGXq0KtfBWIMgFfoYW7co8CDL2RL4LorLf5H_NqVv9LcXYniN7iphFJMYqVGUMjtTVcIguB9w&state=5f3638c6-43f0-41b3-b75c-4be5c4883601

✅ Authorization code and state extracted successfully.

➡️ Proceed to Cell 4 to complete authentication.
------------------------------------------------------------


In [24]:
# -*- coding: utf-8 -*-
"""
LinkedIn Post Helper - Cell 4: Complete Authentication

Uses the code/state extracted in Cell 3 to exchange them for an access token
and fetch the necessary User URN. Confirms if authentication is successful.
"""
print("STEP 2: Completing Authentication")
print("-" * 60)

# --- Pre-check and Attempt Authentication ---
authentication_success = False
# Check if values were successfully extracted in Cell 3
if 'auth_code_from_redirect' in globals() and auth_code_from_redirect and \
   'state_from_redirect' in globals() and state_from_redirect:
    print("Attempting authentication (Token Exchange & URN Fetch)...")
    # Call the auth function defined in Cell 1
    authentication_success = exchange_code_for_token(auth_code_from_redirect, state_from_redirect)

    if authentication_success:
        print("\n✅ Authentication Complete: Token and URN obtained successfully.")
        print("\n➡️ Proceed to Cell 5 to compose and post your update.")
    else:
        print("\n🔴 Authentication Failed. Review errors above.")
        print("   You may need to restart the process from Cell 2.")
        # Auth function handles clearing token/URN state on failure
else:
    print("🔴 Cannot authenticate: Code or state missing from Cell 3 execution.")
    print("   Please run Cell 2 & 3 again successfully.")

# --- End of Cell ---
print("-" * 60)
# Clear the temporary code/state now that this step is done
auth_code_from_redirect = None
state_from_redirect = None
# Note: %store variable state_value_to_send is cleaned up inside exchange_code_for_token

STEP 2: Completing Authentication
------------------------------------------------------------
Attempting authentication (Token Exchange & URN Fetch)...
   Verifying state...
   ✅ State verified.
   Requesting access token...
   ✅ Access Token retrieved!
   Fetching User Info via /v2/userinfo...
   ✅ User URN constructed: urn:li:person:Zov6SporrM

✅ Authentication Complete: Token and URN obtained successfully.

➡️ Proceed to Cell 5 to compose and post your update.
------------------------------------------------------------


In [25]:
# -*- coding: utf-8 -*-
"""
LinkedIn Post Helper - Cell 5: Compose and Post Update

If authentication in Cell 4 was successful, this cell prompts for the
post text and posts it to LinkedIn (Visibility: CONNECTIONS).
"""
print("STEP 3: Compose and Post Update")
print("-" * 60)

# --- Check Authentication State and Proceed to Post ---
# Verify that Cell 4 ran successfully by checking token and URN
if 'access_token' in globals() and access_token and \
   'user_urn' in globals() and user_urn:

    print("✅ Authentication successful. Ready to compose post (Visibility: CONNECTIONS)...")

    # Get post content
    post_text = input(f"Enter the text for your post:\n")

    if post_text:
        print("\nAttempting to post...")
        # Call the posting function defined in Cell 1
        post_linkedin_update(post_text) # Defaults to CONNECTIONS visibility
    else:
        print("⚠️ No text provided. Post cancelled.")

else:
    # Guide user if auth failed or Cell 4 wasn't run/successful
    print("🔴 Cannot post: Authentication incomplete or failed.")
    print("   Please ensure Cell 4 completed successfully before running this cell.")
    if 'access_token' not in globals() or not access_token: print("   (Hint: Access Token is missing)")
    if 'user_urn' not in globals() or not user_urn: print("   (Hint: User URN is missing)")

# --- End of Cell ---
print("-" * 60)

STEP 3: Compose and Post Update
------------------------------------------------------------
✅ Authentication successful. Ready to compose post (Visibility: CONNECTIONS)...
Enter the text for your post:
created text

Attempting to post...
   Posting update (Visibility: CONNECTIONS)...
✅ Successfully posted!
   View Post: https://www.linkedin.com/feed/update/urn:li:share:7324205532242747392/
------------------------------------------------------------


In [26]:
# -*- coding: utf-8 -*-
"""
LinkedIn Post Helper - Cell 6 (Optional): Clear Session State

Clears the stored access token and User URN. Useful for testing
error handling or manually forcing re-authentication on the next run.
"""
print("Clearing session state (token, URN)...")
access_token = None
user_urn = None
# Optional: Clear stored state variable too (should be cleared automatically though)
try: get_ipython().run_line_magic('store', '-d state_value_to_send');
except Exception: pass

print("✅ Session state cleared.")
print("\nYou will need to re-authenticate starting from Cell 2 to post again.")

Clearing session state (token, URN)...
✅ Session state cleared.

You will need to re-authenticate starting from Cell 2 to post again.
